### Data Ingestion

In [61]:
loader = TextLoader("C:\\Users\\User\\RAG_pratice\\data\\text\\Kist.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': 'C:\\Users\\User\\RAG_pratice\\data\\text\\Kist.txt'}, page_content='KIST COLLEGE & SS, KATHMANDU — BACHELOR-LEVEL KNOWLEDGE BASE\nPrepared for RAG / Data Science Practice\nInformation checked against KIST College official web pages available in August 2026.\n\nIMPORTANT DATA-QUALITY NOTE\nThis document is designed as a reliable retrieval corpus, not as an admission contract.\nKIST publishes current programmes and prospectuses, but the exact 2026 rupee-by-rupee fee tables were not exposed in the publicly indexed text pages I could verify. Therefore, no unsupported fee amounts have been invented here. The fee section explicitly separates VERIFIED information from information that must be obtained from the current KIST fee sheet/admission office. This is important for RAG evaluation: the system should answer "not publicly verified" rather than hallucinating a fee.\n\n============================================================\n1. BASIC INSTITUTIONAL INFORMA

## RAG learing progress

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("API key loaded:", os.getenv("GROQ_API_KEY") is not None)

API key loaded: True


In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq initialized successfully!")

Groq initialized successfully!


In [3]:
response = llm.invoke(
    "What is artificial intelligence?"
)

print(response.content)

Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

AI technology is based on the principle of creating algorithms that can process data, identify patterns, and make decisions without being explicitly programmed for each specific task. This allows AI systems to adapt and improve over time, much like humans do.

There are several key characteristics of artificial intelligence:

1. **Machine learning**: AI systems can learn from data and improve their performance over time.
2. **Reasoning**: AI systems can draw conclusions and make decisions based on the data they have been trained on.
3. **Problem-solving**: AI systems can identify and solve complex problems.
4. **Natural language processing**: AI systems can understand and generate human language.
5. **Computer vision*

In [ ]:
def split_document(document, chunk_size=1000, chunk_overlap=200):
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", "","\n============================================================\n"]
    )

    chunks = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(chunks)} chunks.")

    # Example of a chunk
    if chunks:
        print("\nExample chunk:")
        print(f"content:{chunks[0].page_content[:200]}")
        print(f"Metadata:{chunks[0].metadata}")

    return chunks

In [71]:
chunk=split_document(document)

Split 1 documents into 28 chunks.

Example chunk:
content:KIST COLLEGE & SS, KATHMANDU — BACHELOR-LEVEL KNOWLEDGE BASE
Prepared for RAG / Data Science Practice
Information checked against KIST College official web pages available in August 2026.

IMPORTANT D
Metadata:{'source': 'C:\\Users\\User\\RAG_pratice\\data\\text\\Kist.txt'}


In [72]:
import langchain_community
print(langchain_community.__version__)


0.4.2


### Embedding and Vector Chunking

In [73]:
import os
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [75]:
class EmbeddingManager:
    """Handels documnet embedding generation using SenntenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """"
        Initilize The embedding manager

        Args:
            model_name : HuggingFace model name for sentence Embeddings
        """
        self.model_name= model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded Successfully, Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise 
    def generate_embeddings(self, texts: List[str]) -> List[np.ndarray]:
        """
        Generate embeddings for a list of documents

        Args:
            texts: List of text strings
        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dimension)
        """
        if not self.model:
            raise ValueError("Model not Loaded")
        print(f"Generating embeddings for {len(texts)} documents")
        embeddings=self.model.encode(texts,show_progress_bar=True) # 
        print(f"Embeddings generated with shape: {embeddings.shape}")
        return embeddings



embedding_manager=EmbeddingManager()
embedding_manager

INFO: No device provided, using cpu


Loading model: all-MiniLM-L6-v2


INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingfac

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/

Model loaded Successfully, Embedding Dimension: 384


C:\Users\User\AppData\Local\Temp\ipykernel_13820\549366673.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded Successfully, Embedding Dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector BD

In [76]:
import os
class VectorStore:

    def __init__(self, collection_name="pdf_documents",
                 persist_directory="../data/vector_store"):

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB"""

        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embedding for RAG"
                }
            )

            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing collections: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documnets(self, documnets: List[Any], embeddings: np.ndarray):
  

        if len(documnets) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same")

        print(f"Adding {len(documnets)} documents to the vector store")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documnets, embeddings)):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

    # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_texts
        )

            print(f"Successfully added {len(documnets)} documents to the vector store")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise    
vectorstore=VectorStore()

Vector store initialized with collection: pdf_documents
Existing collections: 76


In [77]:
chunk

[Document(metadata={'source': 'C:\\Users\\User\\RAG_pratice\\data\\text\\Kist.txt'}, page_content='KIST COLLEGE & SS, KATHMANDU — BACHELOR-LEVEL KNOWLEDGE BASE\nPrepared for RAG / Data Science Practice\nInformation checked against KIST College official web pages available in August 2026.\n\nIMPORTANT DATA-QUALITY NOTE\nThis document is designed as a reliable retrieval corpus, not as an admission contract.\nKIST publishes current programmes and prospectuses, but the exact 2026 rupee-by-rupee fee tables were not exposed in the publicly indexed text pages I could verify. Therefore, no unsupported fee amounts have been invented here. The fee section explicitly separates VERIFIED information from information that must be obtained from the current KIST fee sheet/admission office. This is important for RAG evaluation: the system should answer "not publicly verified" rather than hallucinating a fee.\n\n============================================================\n1. BASIC INSTITUTIONAL INFORMA

In [78]:
# converting text to embeddings

text=[doc.page_content for doc in chunk]

# Now generating embedding
embeddings=embedding_manager.generate_embeddings(text)

# store in Vector DB
vectorstore.add_documnets(chunk, embeddings)

Generating embeddings for 28 documents


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated with shape: (28, 384)
Adding 28 documents to the vector store
Successfully added 28 documents to the vector store


In [79]:
print(len(chunk))
print(len(embeddings))

28
28


### Retriever pipeline From VectorStore

In [89]:
class RAGRetriver:
    "Handles query-based retrieval from the vector store"

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve( self, query: str, top_k: int = 5, score_threshold: float=0.0)-> List[Dict[str, Any]]:
        """ 
        Retrive relevent documnet for a query 

        Args: 
            query: The input query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score for retrieval 

        Returns:
            List of dictionaries containing document content, metadata, and similarity score
        """


        print(f"Retrieving document for query: '{query}'")
        print(f"Top k: {top_k},score threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Searching in vector store

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #process_result
            retrieved_docs=[]

            if results["documents"] and results["documents"][0]:
                documents=results["documents"][0]
                metadatas = results["metadatas"][0]
                distances=results["distances"][0]
                ids=results["ids"][0]

                print("Distances:",distances)

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similarity score  
                    similarity_score=1-distance

                    
                    retrieved_docs.append({
                           # "id":doc_id,
                            "content":document,
                            #"metadata":metadata,
                           # "similarity_score":similarity_score,
                           # "rank":i+1

                        })

                print(f"Retrived {len(retrieved_docs)} documnets(after filtering)")

            else:
                print("No documnets found")

            return retrieved_docs


        except Exception as e:
            print(f"Error during retrieval:{e}")
            return []


rag_retriever=RAGRetriver(vectorstore,embedding_manager)


In [90]:
rag_retriever.retrieve("BIT")

Retrieving document for query: 'BIT'
Top k: 5,score threshold: 0.0
Generating embeddings for 1 documents


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated with shape: (1, 384)
Distances: [1.5344974994659424, 1.5524075031280518, 1.6458622217178345, 1.6790270805358887, 1.6843602657318115]
Retrived 5 documnets(after filtering)


[{'content': 'Core purpose:\nBIT develops professional IT knowledge and skills in software, hardware, networking, data structures, computer architecture and information systems.\n\nMajor knowledge areas:\n- Programming\n- Software development\n- Hardware concepts\n- Networking\n- Data structures\n- Computer architecture\n- Web development\n- Database development\n- System modelling\n- Computer security\n- Mobile applications\n- Desktop applications\n- Information systems\n\nCareer areas:\n- Software development\n- Web development\n- Mobile development\n- Database administration\n- Network administration\n- Information security\n- Systems analysis\n- IT support\n- IT project work\n- Entrepreneurship\n\nKIST highlights:\n- Network labs\n- Multimedia resources\n- E-library\n- Project work\n- Workshops\n- Guest lectures\n- Industry tie-ups\n- Placement support\n- Entrepreneurship Cell\n- Research and Development Unit'},
 {'content': 'Admission:\n- Minimum D in theory and C in practical in 